# Fairness Diagnostics for Recidivism Risk Prediction

This notebook studies how far a recidivism-risk classifier trained on the
[COMPAS](https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing)
dataset can be pushed toward group fairness, and which features actually drive
its disparities. Two complementary diagnostics are used:

1. **Accuracy-constrained threshold optimization** – how much predictive
   accuracy must be traded away to reach a target level of demographic parity
   (the p-rule), for race, sex, and age as the protected attribute.
2. **Information-theoretic feature selection (FFS)** – decomposing mutual
   information between features, the outcome, and a protected attribute to
   attribute each feature's contribution to accuracy versus discrimination,
   via Shapley values.

Method (1) is adapted from the disparate-impact-removal approach in
[Zafar et al., *Fairness Constraints*](https://github.com/mbilalzafar/fair-classification/tree/master/disparate_impact).
Method (2) implements the feature-selection criteria from
[Information Theoretic Measures for Fairness-aware Feature Selection](https://arxiv.org/abs/2106.00772).

## Setup

In [1]:
import itertools
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 4)

RANDOM_STATE = 42
TARGET = "two_year_recid"

In [2]:
data = pd.read_csv("../data/compas-scores-two-years.csv")
data.shape

(7214, 53)

Shared preprocessing: COMPAS race is restricted to African-American and
Caucasian defendants (the two groups with enough support for a reliable
comparison), jail stay is binned into short/medium/long, and prior-offense
counts are binned, following the original ProPublica analysis conventions.

In [3]:
def encode_features(df):
    race_mapping = {"African-American": 0, "Caucasian": 1}
    sex_mapping = {"Male": 1, "Female": 0}
    age_cat_mapping = {"Less than 25": 0, "25 - 45": 1, "Greater than 45": 2}
    charge_degree_mapping = {"F": 0, "M": 1}

    df = df[df["race"].isin(race_mapping)].copy()
    df["race"] = df["race"].map(race_mapping)
    df["sex"] = df["sex"].map(sex_mapping)
    df["age_cat"] = df["age_cat"].map(age_cat_mapping)
    df["c_charge_degree"] = df["c_charge_degree"].map(charge_degree_mapping)
    return df


def add_length_of_stay(df):
    df = df.copy()
    jail_in = pd.to_datetime(df["c_jail_in"])
    jail_out = pd.to_datetime(df["c_jail_out"])
    stay_days = (jail_out - jail_in).dt.days
    df["length_of_stay"] = stay_days.apply(lambda d: 0 if d <= 7 else (2 if d > 90 else 1))
    return df


def bin_priors_count(df):
    df = df.copy()
    df["priors_count"] = df["priors_count"].apply(lambda c: 0 if c == 0 else (2 if c > 3 else 1))
    return df


def p_rule(protected_rate, non_protected_rate):
    return min(protected_rate / non_protected_rate, non_protected_rate / protected_rate) * 100

## Part 1 – Accuracy-Constrained Fairness Optimization

A logistic regression classifier is trained without any fairness constraint,
then its decision threshold is swept to find the fairest threshold that still
meets an accuracy budget of `initial_accuracy * (1 - gamma)`. `gamma` is the
maximum fraction of accuracy the analyst is willing to give up in exchange for
fairness; sweeping it traces out the accuracy/fairness frontier.

Fairness here is measured with the **p-rule**, the ratio of positive-outcome
rates between the protected and non-protected groups (100% = parity), and
with the covariance between the protected attribute and the predicted label.

### Race as the protected attribute

In [4]:
race_source_columns = [
    "age", "c_charge_degree", "age_cat", "sex",
    "priors_count", "is_recid", "c_jail_in", "c_jail_out",
]
SENSITIVE_ATTR = "race"

race_data = encode_features(data[race_source_columns + [TARGET, SENSITIVE_ATTR]])
race_data = add_length_of_stay(race_data)
race_data = bin_priors_count(race_data)

model_features = ["age_cat", "c_charge_degree", "sex", "priors_count", "length_of_stay"]
X, y = race_data[model_features], race_data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE
)

clf_race = LogisticRegression(solver="liblinear").fit(X_train, y_train)
y_pred_race = clf_race.predict(X_test)

test_race = pd.concat([X_test, race_data.loc[X_test.index, SENSITIVE_ATTR], y_test], axis=1)
protected_race = (test_race[SENSITIVE_ATTR] == 0).to_numpy()

accuracy_race = accuracy_score(y_test, y_pred_race)
p_rule_race = p_rule(y_pred_race[protected_race].mean(), y_pred_race[~protected_race].mean())
covariance_race = np.cov(protected_race, y_pred_race)[0, 1]

print(f"Unconstrained accuracy: {accuracy_race:.3f}")
print(f"Unconstrained p-rule (race): {p_rule_race:.1f}%")
print(f"Covariance(race, prediction): {covariance_race:.4f}")

Unconstrained accuracy: 0.660
Unconstrained p-rule (race): 48.8%
Covariance(race, prediction): 0.0591


The unconstrained model's p-rule sits well below the commonly cited 80%
fairness threshold, and the positive covariance between race and the
predicted label confirms the classifier reproduces the disparity already
present in the training labels.

In [5]:
def optimize_threshold_for_fairness(model, X_test, y_test, protected, gamma=0.0, n_thresholds=1001):
    """Sweep decision thresholds and keep the one with the best p-rule
    that still meets an accuracy budget of (1 - gamma) times baseline."""
    baseline_accuracy = accuracy_score(y_test, model.predict(X_test))
    target_accuracy = baseline_accuracy * (1 - gamma)
    probabilities = model.predict_proba(X_test)[:, 1]

    best = {"threshold": 0.5, "accuracy": baseline_accuracy, "p_rule": 0.0, "covariance": float("inf")}

    for threshold in np.linspace(0, 1, n_thresholds):
        y_pred = (probabilities >= threshold).astype(int)
        accuracy = accuracy_score(y_test, y_pred)
        if accuracy < target_accuracy:
            continue

        protected_rate = y_pred[protected].mean()
        non_protected_rate = y_pred[~protected].mean()
        if non_protected_rate == 0:
            continue

        current_p_rule = p_rule(protected_rate, non_protected_rate)
        current_covariance = np.cov(protected, y_pred)[0, 1]

        is_better = current_p_rule > best["p_rule"] or (
            current_p_rule == best["p_rule"] and abs(current_covariance) < abs(best["covariance"])
        )
        if is_better:
            best = {
                "threshold": threshold,
                "accuracy": accuracy,
                "p_rule": current_p_rule,
                "covariance": current_covariance,
            }

    return best


optimize_threshold_for_fairness(clf_race, X_test, y_test, protected_race, gamma=0.0)

{'threshold': np.float64(0.464),
 'accuracy': 0.6644986449864498,
 'p_rule': np.float64(54.87236695321215),
 'covariance': np.float64(0.06064053048339596)}

In [6]:
gamma_sweep = []
for gamma_value in np.arange(0, 0.26, 0.01):
    result = optimize_threshold_for_fairness(clf_race, X_test, y_test, protected_race, gamma=gamma_value)
    gamma_sweep.append({"gamma": gamma_value, **result})

gamma_sweep = pd.DataFrame(gamma_sweep)
gamma_sweep.head()

,gamma,threshold,accuracy,p_rule,covariance
0,0.00,0.464,0.6645,54.8724,0.0606
1,0.01,0.456,0.6547,62.6504,0.0554
2,0.02,0.438,0.6472,63.8291,0.0566
3,0.03,0.438,0.6472,63.8291,0.0566
4,0.04,0.438,0.6472,63.8291,0.0566


In [7]:
fig, accuracy_axis = plt.subplots()

accuracy_axis.set_xlabel("Gamma")
accuracy_axis.set_ylabel("Accuracy", color="tab:blue")
accuracy_axis.plot(gamma_sweep["gamma"], gamma_sweep["accuracy"], color="tab:blue")
accuracy_axis.tick_params(axis="y", labelcolor="tab:blue")

p_rule_axis = accuracy_axis.twinx()
p_rule_axis.set_ylabel("P-rule", color="tab:red")
p_rule_axis.plot(gamma_sweep["gamma"], gamma_sweep["p_rule"], color="tab:red")
p_rule_axis.tick_params(axis="y", labelcolor="tab:red")

fig.tight_layout()
plt.title("Accuracy and P-rule vs. Gamma (race)")
plt.savefig("../figures/accuracy_prule_vs_gamma.png", dpi=150, bbox_inches="tight")
plt.show()

As `gamma` increases, accuracy decreases monotonically while the p-rule
climbs toward 100%. Reaching an 80% p-rule costs roughly ten points of
accuracy relative to the unconstrained baseline – a fairness/accuracy
trade-off that is explicit and tunable rather than implicit and hidden, which
is the main practical value of this method.

### Sex as the protected attribute

In [8]:
SENSITIVE_ATTR = "sex"
sex_features = ["age", "priors_count"]

sex_dummies = pd.get_dummies(data[sex_features + [SENSITIVE_ATTR]])
X_sex = sex_dummies.drop(columns=[f"{SENSITIVE_ATTR}_Female", f"{SENSITIVE_ATTR}_Male"])
y_sex = data[TARGET]

X_train_sex, X_test_sex, y_train_sex, y_test_sex = train_test_split(
    X_sex, y_sex, test_size=0.3, random_state=RANDOM_STATE
)

clf_sex = LogisticRegression(solver="liblinear").fit(X_train_sex, y_train_sex)
y_pred_sex = clf_sex.predict(X_test_sex)

test_sex = pd.concat([X_test_sex, data.loc[X_test_sex.index, SENSITIVE_ATTR], y_test_sex], axis=1)
protected_sex = (test_sex[SENSITIVE_ATTR] == "Female").to_numpy()

accuracy_sex = accuracy_score(y_test_sex, y_pred_sex)
p_rule_sex_baseline = p_rule(y_pred_sex[protected_sex].mean(), y_pred_sex[~protected_sex].mean())

print(f"Unconstrained accuracy: {accuracy_sex:.3f}")
print(f"Unconstrained p-rule (sex): {p_rule_sex_baseline:.1f}%")

fairness_optimized_sex = optimize_threshold_for_fairness(
    clf_sex, X_test_sex, y_test_sex, protected_sex, gamma=0.2, n_thresholds=101
)
fairness_optimized_sex

Unconstrained accuracy: 0.685
Unconstrained p-rule (sex): 61.1%


{'threshold': np.float64(0.3),
 'accuracy': 0.5547344110854503,
 'p_rule': np.float64(94.85804669182075),
 'covariance': np.float64(-0.006300452929098031)}

The unconstrained disparity by sex is smaller than by race, and a modest
accuracy budget (`gamma = 0.2`) is enough to push the p-rule for sex close to
parity.

### Age category as the protected attribute

In [9]:
SENSITIVE_ATTR = "age_cat_mapped"
age_cat_mapping = {"Less than 25": 0, "25 - 45": 1, "Greater than 45": 2}
data_age = data.assign(age_cat_mapped=data["age_cat"].map(age_cat_mapping))

age_features = ["sex", "priors_count", SENSITIVE_ATTR]
age_dummies = pd.get_dummies(data_age[age_features])
X_age = age_dummies.drop(columns=[SENSITIVE_ATTR])
y_age = data_age[TARGET]

X_train_age, X_test_age, y_train_age, y_test_age = train_test_split(
    X_age, y_age, test_size=0.3, random_state=RANDOM_STATE
)

clf_age = LogisticRegression(solver="liblinear").fit(X_train_age, y_train_age)
y_pred_age = clf_age.predict(X_test_age)

test_age = pd.concat([X_test_age, data_age.loc[X_test_age.index, SENSITIVE_ATTR], y_test_age], axis=1)
accuracy_age = accuracy_score(y_test_age, y_pred_age)

baseline_by_group = []
for age_group in age_cat_mapping.values():
    protected = (test_age[SENSITIVE_ATTR] == age_group).to_numpy()
    group_p_rule = p_rule(y_pred_age[protected].mean(), y_pred_age[~protected].mean())
    group_covariance = np.cov(protected, y_pred_age)[0, 1]
    baseline_by_group.append({"age_group": age_group, "p_rule": group_p_rule, "covariance": group_covariance})

print(f"Unconstrained accuracy: {accuracy_age:.3f}")
pd.DataFrame(baseline_by_group)

Unconstrained accuracy: 0.648


,age_group,p_rule,covariance
0,0,32.8507,-0.0326
1,1,65.0254,0.0247
2,2,83.1494,0.0079


In [10]:
fairness_optimized_by_age = []
for age_group in age_cat_mapping.values():
    protected = (test_age[SENSITIVE_ATTR] == age_group).to_numpy()
    result = optimize_threshold_for_fairness(clf_age, X_test_age, y_test_age, protected, gamma=0.15, n_thresholds=101)
    fairness_optimized_by_age.append({"age_group": age_group, **result})

pd.DataFrame(fairness_optimized_by_age)

,age_group,threshold,accuracy,p_rule,covariance
0,0,0.39,0.5982,80.3259,-0.0217
1,1,0.79,0.5797,98.6782,0.0001
2,2,0.37,0.5940,97.9257,0.0022


The unconstrained model is least fair to the youngest defendants (age group
`0`, under 25), and a `gamma = 0.15` accuracy budget disproportionately
improves the p-rule for that group – accuracy-constrained thresholding does
not just shift the average disparity, it can be targeted at the
worst-affected subgroup.

**Part 1 summary:** all three protected attributes show a real
accuracy/fairness trade-off, but the size of the trade-off differs sharply –
race requires the largest accuracy sacrifice to reach a given p-rule, age
category the most subgroup-heterogeneous effect, and sex the smallest
correction. This threshold-based approach corrects a trained model's
*decision boundary* without touching its features. Part 2 asks whether the
same disparities can instead be addressed at the *feature* level.

## Part 2 – Information-Theoretic Fairness-Aware Feature Selection

Accuracy-constrained thresholding treats the model as a black box. This
section instead asks *which features* carry information that is useful for
prediction versus information that is merely correlated with a protected
attribute, using a mutual-information decomposition.

**Bivariate decomposition of mutual information:**

$$I(T; R_1, R_2) = UI(T; R_1 \setminus R_2) + UI(T; R_2 \setminus R_1) + SI(T; R_1, R_2) + CI(T; R_1, R_2)$$

the mutual information between outcome $T$ and two variables $R_1, R_2$ decomposes into
unique information from $R_1$ alone, unique information from $R_2$ alone, information
shared between them, and synergistic information only visible when both are known together.

Generalized to one variable at a time:

$$I(T; R_i) = UI(T; R_i \setminus R_j) + SI(T; R_1, R_2), \quad i \neq j$$

Here $T$ is two-year recidivism and $R_i$ is race (African-American or Caucasian).

### Preprocessing

In [11]:
ffs_data = encode_features(data)
ffs_data = add_length_of_stay(ffs_data)
ffs_data = bin_priors_count(ffs_data)
final_dataset = ffs_data.drop_duplicates()

print("Raw shape:", data.shape)
print("After race filter + dedup:", final_dataset.shape)

Raw shape: (7214, 53)
After race filter + dedup: (6150, 54)


### Mutual information scores (bivariate, accuracy coefficient, discriminatory effect)

In [12]:
race_recidivism_data = data[data["race"].isin(["Caucasian", "African-American"])].copy()
race_recidivism_data["race"] = race_recidivism_data["race"].map({"Caucasian": 1, "African-American": 0})

mi_race = mutual_info_score(race_recidivism_data[TARGET], race_recidivism_data["race"])
print("Mutual information between recidivism and race:", mi_race)

Mutual information between recidivism and race: 0.007054417358854759


This value is close to zero, meaning race alone is nearly independent of the
recidivism outcome – on its own, race is not a strongly predictive feature.

**Accuracy coefficient** $vAcc$: for a feature subset $X_S$ and its
complement $X_{S_C}$,

$$vAcc(X_S) = I(Y; X_S \mid X_{S_C}) = UI(Y; X_S \setminus X_{S_C}) + CI(Y; X_S, X_{S_C})$$

the conditional mutual information between the outcome and $X_S$ given
everything else – i.e. the information $X_S$ adds on top of the rest of the
feature set.

In [13]:
def conditional_mutual_info(df, target_col, feature_cols, condition_cols):
    if not condition_cols:
        return mutual_info_score(df[target_col], df[feature_cols])
    joint_cols = [feature_cols] + condition_cols if isinstance(feature_cols, str) else feature_cols + condition_cols
    joint = df[joint_cols].apply(lambda row: "_".join(row.values.astype(str)), axis=1)
    condition_only = df[condition_cols].apply(lambda row: "_".join(row.values.astype(str)), axis=1)
    return mutual_info_score(df[target_col], joint) - mutual_info_score(df[target_col], condition_only)


cmi_race_alone = conditional_mutual_info(race_recidivism_data, TARGET, "race", [])
cmi_race_given_age_sex = conditional_mutual_info(race_recidivism_data, TARGET, "race", ["age", "sex"])

print("MI(recidivism, race):", cmi_race_alone)
print("CMI(recidivism, race | age, sex):", cmi_race_given_age_sex)

MI(recidivism, race): 0.007054417358854759
CMI(recidivism, race | age, sex): 0.012090642380671007


Because there is no conditioning set in the first call, the conditional
mutual information reduces to the bivariate score. Conditioning on age and
sex raises the informativeness of race by about 70%, meaning the combination
of race with age and sex tells us more about recidivism than race alone – but
both values remain small in absolute terms.

**Discriminatory effect** $vD$: for a feature subset $X_S$ and protected
attribute $A$,

$$v^D(X_S) = SI(Y; X_S, A) \times I(X_S; A) \times I(X_S; A \mid Y)$$

the product of how much $X_S$'s information about $Y$ is *shared* with $A$,
how much $X_S$ itself correlates with $A$, and how much that correlation
persists once $Y$ is known.

In [14]:
def shared_information(df, target_col, feature_cols, protected_col):
    if isinstance(feature_cols, str):
        feature_cols = [feature_cols]
    mi_target_protected = mutual_info_score(df[target_col], df[protected_col])
    joint = df[feature_cols + [protected_col]].apply(lambda row: "_".join(row.values.astype(str)), axis=1)
    mi_target_joint = mutual_info_score(df[target_col], joint)
    unique_info = mi_target_protected - mi_target_joint
    return max(mi_target_protected - unique_info, 0)


si_race = shared_information(race_recidivism_data, TARGET, "race", "race")
mi_race_protected = mutual_info_score(race_recidivism_data["race"], race_recidivism_data["race"])
vD_race = si_race * mi_race_protected * cmi_race_alone
print("Discriminatory effect (vD) of race:", vD_race)

age_sex_joint = race_recidivism_data[["age", "sex"]].apply(lambda row: "_".join(row.values.astype(str)), axis=1)
si_age_sex = shared_information(race_recidivism_data, TARGET, ["age", "sex"], "race")
mi_age_sex_race = mutual_info_score(age_sex_joint, race_recidivism_data["race"])
vD_age_sex = si_age_sex * mi_age_sex_race * cmi_race_given_age_sex
print("Discriminatory effect (vD) of age & sex:", vD_age_sex)

Discriminatory effect (vD) of race: 3.347250942628674e-05
Discriminatory effect (vD) of age & sex: 1.9407422695348022e-05


Both discriminatory-effect values are small, and `age`/`sex` together carry a
smaller discriminatory signal than `race` alone – consistent with the
accuracy-coefficient result above. Race is the single most influential
feature for predicting recidivism among the ones examined, but the overall
effect sizes are modest, which suggests unmeasured factors outside this
dataset likely matter more than any one demographic feature.

### Choosing a baseline feature set

In [15]:
core_features = ["c_charge_degree", "age_cat", "sex", "race", "length_of_stay"]
extended_features = ["priors_count"] + core_features

shuffled = final_dataset.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
n = len(shuffled)
train_set = shuffled.iloc[: int(n * 5 / 7)]
test_set = shuffled.iloc[int(n * 5 / 7) : int(n * 6 / 7)]
validation_set = shuffled.iloc[int(n * 6 / 7) :]

train_data_core, train_data_extended, train_labels = (
    train_set[core_features], train_set[extended_features], train_set[TARGET].to_numpy(),
)
test_data_core, test_data_extended, test_labels = (
    test_set[core_features], test_set[extended_features], test_set[TARGET].to_numpy(),
)
validation_data_core, validation_data_extended, validation_labels = (
    validation_set[core_features], validation_set[extended_features], validation_set[TARGET].to_numpy(),
)

clf_with_priors = LogisticRegression().fit(train_data_extended, train_labels)
clf_without_priors = LogisticRegression().fit(train_data_core, train_labels)

accuracy_with_priors = clf_with_priors.score(validation_data_extended, validation_labels)
accuracy_without_priors = clf_without_priors.score(validation_data_core, validation_labels)

print(f"Validation accuracy with priors_count:    {accuracy_with_priors:.3f}")
print(f"Validation accuracy without priors_count: {accuracy_without_priors:.3f}")

Validation accuracy with priors_count:    0.659
Validation accuracy without priors_count: 0.605


`priors_count` measurably improves validation accuracy, so it is kept in the
feature set used for the rest of the analysis, even though prior-offense
counts are themselves correlated with policing intensity and, transitively,
with race.

### Shapley-value decomposition of feature contributions

To go from a single feature to a per-feature score, each characteristic
function ($vAcc$, $vD$) is turned into a Shapley value – the standard way in
cooperative game theory to fairly split a joint quantity among individual
contributors. Given a power set $P$ and characteristic function
$v(\cdot): P([n]) \to \mathbb{R}$:

$$\varphi_i = \sum_{T \subseteq [n] \setminus i} \frac{|T|!\,(n-|T|-1)!}{n!} \big(v(T \cup \{i\}) - v(T)\big), \quad \forall i \in [n]$$

Applying this to $vAcc$ and $vD$ gives the **marginal accuracy coefficient**
$\varphi_{Acc}$ and **marginal discrimination coefficient** $\varphi_D$ for
each feature.

In [16]:
def unique_values_per_column(array):
    return [np.unique(array[:, col]).tolist() for col in range(array.shape[1])]


def unique_information(array_1, array_2):
    combined = np.concatenate((array_1, array_2), axis=1)
    total_info = 0.0
    for combo in itertools.product(*unique_values_per_column(combined)):
        joint_rate = np.mean((combined == combo).all(axis=1))
        rate_1 = np.mean((array_1 == combo[: array_1.shape[1]]).all(axis=1))
        rate_2 = np.mean((array_2 == combo[array_1.shape[1] :]).all(axis=1))
        if joint_rate > 0 and rate_1 > 0 and rate_2 > 0:
            total_info += abs(joint_rate * np.log(joint_rate / rate_1) / rate_1)
    return total_info


def unique_information_conditional(array_1, array_2, condition):
    combined = np.concatenate((array_1, array_2, condition), axis=1)
    total_info = 0.0
    for combo in itertools.product(*unique_values_per_column(combined)):
        joint_mask = (combined == combo).all(axis=1)
        mask_1 = (array_1 == combo[: array_1.shape[1]]).all(axis=1)
        mask_2_given_condition = (combined[:, array_1.shape[1] :] == combo[array_1.shape[1] :]).all(axis=1)

        joint_rate = np.mean(joint_mask)
        rate_1 = np.mean(mask_1)
        rate_2 = np.mean(mask_2_given_condition)
        rate_1_given_2 = np.mean(mask_1[mask_2_given_condition]) if np.any(mask_2_given_condition) else 0

        if joint_rate > 0 and rate_1 > 0 and rate_2 > 0 and rate_1_given_2 > 0:
            total_info += abs(joint_rate * np.log(joint_rate / rate_2) / rate_1_given_2)
    return total_info


def accuracy_coef(outcome, feature_subset, complement, protected):
    condition = np.concatenate((complement, protected), axis=1)
    return unique_information_conditional(outcome, feature_subset, condition)


def discrimination_coef(outcome, feature_subset, protected):
    feature_and_protected = np.concatenate((feature_subset, protected), axis=1)
    return (
        unique_information(outcome, feature_and_protected)
        * unique_information(feature_subset, protected)
        * unique_information_conditional(feature_subset, protected, outcome)
    )


def marginal_accuracy_coef(outcome, features, protected, target_feature_idx):
    n_features = features.shape[1]
    value = 0.0
    other_indices = [i for i in range(n_features) if i != target_feature_idx]
    for subset_size in range(n_features):
        for subset in itertools.combinations(other_indices, subset_size):
            weight = math.factorial(len(subset)) * math.factorial(n_features - len(subset) - 1) / math.factorial(n_features)
            with_feature = list(subset) + [target_feature_idx]
            complement_with = [i for i in range(n_features) if i not in with_feature]
            complement_without = [i for i in range(n_features) if i not in subset]
            v_with = accuracy_coef(outcome, features[:, with_feature], features[:, complement_with], protected)
            v_without = accuracy_coef(outcome, features[:, subset], features[:, complement_without], protected)
            value += weight * (v_with - v_without)
    return value


def marginal_discrimination_coef(outcome, features, protected, target_feature_idx):
    n_features = features.shape[1]
    value = 0.0
    other_indices = [i for i in range(n_features) if i != target_feature_idx]
    for subset_size in range(n_features):
        for subset in itertools.combinations(other_indices, subset_size):
            weight = math.factorial(len(subset)) * math.factorial(n_features - len(subset) - 1) / math.factorial(n_features)
            with_feature = list(subset) + [target_feature_idx]
            v_with = discrimination_coef(outcome, features[:, with_feature], protected)
            v_without = discrimination_coef(outcome, features[:, subset], protected)
            value += weight * (v_with - v_without)
    return value


def fairness_utility_score(accuracy_coefs, discrimination_coefs, alpha):
    """Higher is better: reward predictive accuracy, penalize discrimination."""
    return accuracy_coefs - alpha * discrimination_coefs

In [17]:
outcome = train_labels.reshape(-1, 1)
features = train_data_extended.to_numpy()
protected = train_data_extended["race"].to_numpy().reshape(-1, 1)

shapley_accuracy, shapley_discrimination = [], []
for feature_idx in range(len(extended_features)):
    shapley_accuracy.append(marginal_accuracy_coef(outcome, features, protected, feature_idx))
    shapley_discrimination.append(marginal_discrimination_coef(outcome, features, protected, feature_idx))

shapley_scores = pd.DataFrame({
    "feature": extended_features,
    "accuracy_coef": shapley_accuracy,
    "discrimination_coef": shapley_discrimination,
})
shapley_scores

,feature,accuracy_coef,discrimination_coef
0,priors_count,0.0000e+00,8132.9813
1,c_charge_degree,2.5905e-17,6255.4909
2,age_cat,9.6219e-17,8070.2766
3,sex,5.9212e-17,6105.4545
4,race,2.7386e-16,-36415.2544
5,length_of_stay,1.1472e-16,7851.0511


In [18]:
for alpha in [1e-6, 1e-5, 1e-4]:
    utility = fairness_utility_score(shapley_scores["accuracy_coef"], shapley_scores["discrimination_coef"], alpha)
    print(f"alpha = {alpha}")
    print(shapley_scores.assign(fairness_utility=utility).to_string(index=False))
    print()

alpha = 1e-06
        feature  accuracy_coef  discrimination_coef  fairness_utility
   priors_count     0.0000e+00            8132.9813           -0.0081
c_charge_degree     2.5905e-17            6255.4909           -0.0063
        age_cat     9.6219e-17            8070.2766           -0.0081
            sex     5.9212e-17            6105.4545           -0.0061
           race     2.7386e-16          -36415.2544            0.0364
 length_of_stay     1.1472e-16            7851.0511           -0.0079

alpha = 1e-05
        feature  accuracy_coef  discrimination_coef  fairness_utility
   priors_count     0.0000e+00            8132.9813           -0.0813
c_charge_degree     2.5905e-17            6255.4909           -0.0626
        age_cat     9.6219e-17            8070.2766           -0.0807
            sex     5.9212e-17            6105.4545           -0.0611
           race     2.7386e-16          -36415.2544            0.3642
 length_of_stay     1.1472e-16            7851.0511          

`race` is the only feature with a *negative* discrimination coefficient in
this decomposition, which is why it is the only feature whose fairness
utility score turns positive as `alpha` grows – every other feature is
penalized more as the discrimination weight increases, while race is
rewarded. This is the FFS criterion's concrete recommendation: at a
sufficiently large `alpha`, race is the first feature this method would flag
for removal.

### Calibration under feature removal

Removing a feature can change the raw accuracy/fairness tally, but a
classifier can also be equally accurate for both groups on average while
still being *miscalibrated* – performing systematically better for one group
than the other. Calibration here is measured as
$|{\rm Accuracy}_{\text{Caucasian}} - {\rm Accuracy}_{\text{African-American}}|$;
smaller is better.

In [19]:
def calibration_gap(protected, y_pred, y_true):
    accuracies = []
    for group in (1, 0):  # 1 = Caucasian, 0 = African-American
        mask = protected == group
        accuracies.append((y_pred[mask] == y_true[mask]).mean())
    return abs(accuracies[0] - accuracies[1])


calibration_results = []
for feature in ["base"] + extended_features:
    train_subset = train_data_extended if feature == "base" else train_data_extended.drop(columns=[feature])
    test_subset = test_data_extended if feature == "base" else test_data_extended.drop(columns=[feature])

    clf = LogisticRegression(random_state=RANDOM_STATE).fit(train_subset, train_labels)
    predictions = clf.predict(test_subset)

    calibration_results.append({
        "removed_feature": feature,
        "accuracy": clf.score(test_subset, test_labels),
        "calibration_gap": calibration_gap(test_data_extended["race"].to_numpy(), predictions, test_labels),
    })

pd.DataFrame(calibration_results)

,removed_feature,accuracy,calibration_gap
0,base,0.6416,0.0312
1,priors_count,0.5973,0.0174
2,c_charge_degree,0.6416,0.0312
3,age_cat,0.6348,0.0330
4,sex,0.6382,0.0159
5,race,0.6428,0.0194
6,length_of_stay,0.6394,0.0272


Accuracy and calibration do not move together, so a feature-selection
decision needs to look at both. Removing `age_cat` is lose-lose here (both
accuracy and calibration get worse). Removing `race` is the standout case:
validation accuracy holds steady – it even ticks up slightly – while the
calibration gap between groups nearly halves. `sex` gives the single largest
calibration improvement, at a modest accuracy cost. `c_charge_degree` barely
moves either metric, suggesting it contributes little to this model.

### Final feature-set comparison

In [20]:
candidate_drops = [
    ["length_of_stay", "sex"],
    ["length_of_stay", "priors_count"],
    ["length_of_stay", "age_cat"],
    ["length_of_stay", "race"],
    ["length_of_stay"],
]

final_comparison = []
for dropped in candidate_drops:
    train_subset = train_data_extended.drop(columns=dropped)
    validation_subset = validation_data_extended.drop(columns=dropped)
    clf = LogisticRegression(random_state=RANDOM_STATE).fit(train_subset, train_labels)
    final_comparison.append({
        "dropped_features": ", ".join(dropped),
        "validation_accuracy": clf.score(validation_subset, validation_labels),
    })

pd.DataFrame(final_comparison)

,dropped_features,validation_accuracy
0,"length_of_stay, sex",0.6564
1,"length_of_stay, priors_count",0.5961
2,"length_of_stay, age_cat",0.6337
3,"length_of_stay, race",0.6564
4,length_of_stay,0.6553


**Part 2 summary:** every candidate above already drops `length_of_stay`,
following the calibration result on the previous page, and then removes one
further feature. Dropping `race` on top ties for the best validation
accuracy – consistent with the calibration finding that race is the one
feature whose removal helps or is neutral on every axis measured (accuracy,
calibration, and the FFS discrimination coefficient) – while dropping
`priors_count` on top is clearly the worst choice, confirming it is the
single most load-bearing feature for accuracy in this dataset.

## Conclusion

Threshold-based accuracy-constrained optimization (Part 1) and
information-theoretic feature selection (Part 2) attack the same
COMPAS disparity from two different points in the pipeline – one reshapes the
decision boundary after training, the other decides what goes into training
in the first place. They agree on the qualitative diagnosis (race carries a
real, if modest, discriminatory signal beyond what age and sex explain, and
is the single feature whose removal is close to a free lunch) but offer
different levers: thresholding gives a continuously tunable accuracy/fairness
trade-off with no retraining, while feature selection gives a smaller set of
one-off, interpretable choices – most notably, drop `race` – that change the
model itself. In practice these are complementary rather than competing – a
deployed system could use FFS to choose a defensible feature set and then use
threshold optimization to tune the remaining accuracy/fairness trade-off to a
policy target.